TELECOM CUSTOMER CHURN PREDICTION

IMPORT LIBRARIES

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import xgboost as xgb

2.LOAD DATA

In [23]:
df = pd.read_csv("C:\\Users\\HP\\Documents\\telecom_churn_data.csv")
df.head()

,customer_id,age,gender,tenure_months,monthly_charges,total_charges,contract_type,internet_service,payment_method,tech_support,online_security,senior_citizen,Churn
0,CUST00001,56.0,Male,48,59.04,NaN,Two year,Fiber optic,Credit card,No,No,0,No
1,CUST00002,69.0,Male,2,107.17,199.58,Month-to-month,DSL,Electronic check,No,No,0,Yes
2,CUST00003,46.0,Male,44,70.03,2755.00,Month-to-month,No,Electronic check,No,Yes,0,No
3,CUST00004,32.0,Male,13,91.79,1196.25,Month-to-month,DSL,Bank transfer,No,No internet service,1,Yes
4,CUST00005,60.0,Female,29,28.62,795.53,Month-to-month,Fiber optic,Credit card,Yes,No,0,Yes


3.DATA INSPECTION

In [24]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       1000 non-null   object 
 1   age               990 non-null    float64
 2   gender            990 non-null    object 
 3   tenure_months     1000 non-null   int64  
 4   monthly_charges   990 non-null    float64
 5   total_charges     970 non-null    float64
 6   contract_type     1000 non-null   object 
 7   internet_service  1000 non-null   object 
 8   payment_method    985 non-null    object 
 9   tech_support      980 non-null    object 
 10  online_security   980 non-null    object 
 11  senior_citizen    1000 non-null   int64  
 12  Churn             1000 non-null   object 
dtypes: float64(3), int64(2), object(8)
memory usage: 101.7+ KB


customer_id          0
age                 10
gender              10
tenure_months        0
monthly_charges     10
total_charges       30
contract_type        0
internet_service     0
payment_method      15
tech_support        20
online_security     20
senior_citizen       0
Churn                0
dtype: int64

4.DATA CLEANING

In [25]:
df = df.drop(columns=["customer_id"])

for col in ["age", "monthly_charges", "total_charges"]:
    df[col] = df[col].fillna(df[col].median())

for col in ["gender", "payment_method", "tech_support", "online_security"]:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum()

age                 0
gender              0
tenure_months       0
monthly_charges     0
total_charges       0
contract_type       0
internet_service    0
payment_method      0
tech_support        0
online_security     0
senior_citizen      0
Churn               0
dtype: int64

5.EXPLORATORY DATA ANALYSIS

In [26]:
print("Overall churn rate:")
print(df["Churn"].value_counts(normalize=True))

print("\nChurn rate by contract type:")
print(df.groupby("contract_type")["Churn"].apply(lambda x: (x == "Yes").mean()).round(3))

print("\nChurn rate by internet service:")
print(df.groupby("internet_service")["Churn"].apply(lambda x: (x == "Yes").mean()).round(3))

Overall churn rate:
Churn
No     0.662
Yes    0.338
Name: proportion, dtype: float64

Churn rate by contract type:
contract_type
Month-to-month    0.434
One year          0.220
Two year          0.211
Name: Churn, dtype: float64

Churn rate by internet service:
internet_service
DSL            0.334
Fiber optic    0.374
No             0.264
Name: Churn, dtype: float64


In [27]:
num_df = df[["age", "tenure_months", "monthly_charges", "total_charges", "senior_citizen"]].copy()
num_df["Churn"] = (df["Churn"] == "Yes").astype(int)
num_df.corr()["Churn"].sort_values(ascending=False)

Churn              1.000000
age               -0.001326
monthly_charges   -0.011404
senior_citizen    -0.056783
total_charges     -0.129404
tenure_months     -0.137596
Name: Churn, dtype: float64

6.ENCODING

In [28]:
target = (df["Churn"] == "Yes").astype(int)
features = df.drop(columns=["Churn"])

cat_cols = features.select_dtypes(include="object").columns.tolist()
num_cols = features.select_dtypes(exclude="object").columns.tolist()

features_encoded = pd.get_dummies(features, columns=cat_cols, drop_first=True)
features_encoded.head()

,age,tenure_months,monthly_charges,total_charges,senior_citizen,gender_Male,contract_type_One year,contract_type_Two year,internet_service_Fiber optic,internet_service_No,payment_method_Credit card,payment_method_Electronic check,payment_method_Mailed check,tech_support_No internet service,tech_support_Yes,online_security_No internet service,online_security_Yes
0,56.0,48,59.04,1858.405,0,True,False,True,True,False,True,False,False,False,False,False,False
1,69.0,2,107.17,199.580,0,True,False,False,False,False,False,True,False,False,False,False,False
2,46.0,44,70.03,2755.000,0,True,False,False,False,True,False,True,False,False,False,False,True
3,32.0,13,91.79,1196.250,1,True,False,False,False,False,False,False,False,False,False,True,False
4,60.0,29,28.62,795.530,0,False,False,False,True,False,True,False,False,False,True,False,False


7.TRAIN_TEST_SPLIT

In [29]:
x_train, x_test, y_train, y_test = train_test_split(
    features_encoded, target, test_size=0.2, random_state=42, stratify=target
)
x_train.shape, x_test.shape

((800, 17), (200, 17))

8.SCALING

In [30]:
scaler = StandardScaler()
x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()
x_train_scaled[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test_scaled[num_cols] = scaler.transform(x_test[num_cols])

9.MODEL TRAINIG AND EVALUATION

In [31]:
def evaluate(name, model, xtr, xte):
    model.fit(xtr, y_train)
    pred = model.predict(xte)
    proba = model.predict_proba(xte)[:, 1]
    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)
    print(f"=== {name} ===")
    print("Accuracy:", round(acc, 4), " ROC-AUC:", round(auc, 4))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    return model, acc, auc

LOGISTIC REGRESSION

In [32]:
lr = LogisticRegression(max_iter=1000)
lr, lr_acc, lr_auc = evaluate("Logistic Regression", lr, x_train_scaled, x_test_scaled)

=== Logistic Regression ===
Accuracy: 0.695  ROC-AUC: 0.6954
[[118  14]
 [ 47  21]]
              precision    recall  f1-score   support

           0       0.72      0.89      0.79       132
           1       0.60      0.31      0.41        68

    accuracy                           0.69       200
   macro avg       0.66      0.60      0.60       200
weighted avg       0.68      0.69      0.66       200



RANDOM FOREST

In [33]:
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf, rf_acc, rf_auc = evaluate("Random Forest", rf, x_train, x_test)

=== Random Forest ===
Accuracy: 0.625  ROC-AUC: 0.6051
[[112  20]
 [ 55  13]]
              precision    recall  f1-score   support

           0       0.67      0.85      0.75       132
           1       0.39      0.19      0.26        68

    accuracy                           0.62       200
   macro avg       0.53      0.52      0.50       200
weighted avg       0.58      0.62      0.58       200



XGBOOST

In [34]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42
)
xgb_model, xgb_acc, xgb_auc = evaluate("XGBoost", xgb_model, x_train, x_test)

=== XGBoost ===
Accuracy: 0.64  ROC-AUC: 0.6117
[[103  29]
 [ 43  25]]
              precision    recall  f1-score   support

           0       0.71      0.78      0.74       132
           1       0.46      0.37      0.41        68

    accuracy                           0.64       200
   macro avg       0.58      0.57      0.58       200
weighted avg       0.62      0.64      0.63       200



10.MODEL COMPARISON

In [35]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [lr_acc, rf_acc, xgb_acc],
    "ROC-AUC": [lr_auc, rf_auc, xgb_auc]
})
results

,Model,Accuracy,ROC-AUC
0,Logistic Regression,0.695,0.695410
1,Random Forest,0.625,0.605114
2,XGBoost,0.640,0.611742


11.FEATURE IMPORTANCE

In [36]:
importances = pd.Series(rf.feature_importances_, index=x_train.columns).sort_values(ascending=False)
importances.head(10)

monthly_charges                 0.179141
total_charges                   0.175642
age                             0.158217
tenure_months                   0.157960
contract_type_One year          0.034878
contract_type_Two year          0.030831
internet_service_Fiber optic    0.029389
tech_support_Yes                0.027697
gender_Male                     0.027499
online_security_Yes             0.024444
dtype: float64